# Predicción de Fuga de Clientes (Churn)

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Predecir abandono de clientes** con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Identificar señales de fuga** a partir de comportamiento transaccional
3. **Calcular engagement score** multi-producto
4. **Segmentar clientes** por riesgo de fuga
5. **Generar recomendaciones de retención** con `CORTEX.COMPLETE()`

---

## Lo Que Construirás

Un **sistema predictivo de churn bancario** que identifica clientes en riesgo:
- Modelo de clasificación con features de engagement y transaccionalidad
- Detección de caída de actividad y reducción de saldos
- Recomendaciones personalizadas de retención con IA
- Dashboard de retención con acciones prioritarias

**Valor de Negocio**: Retener clientes de alto valor antes de que abandonen (el coste de adquisición es 5-7x el de retención).

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex ML y Cortex AI
- **Aproximadamente 15 minutos** para completar

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Predicción de churn
- `CORTEX.COMPLETE()` — Recomendaciones de retención
- Window functions (`LAG`, `AVG OVER`) — Tendencias de actividad
- `NTILE()` — Segmentación por cuartiles

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Retención de Clientes

**Objetivo**: Identificar clientes con alta probabilidad de abandonar el banco en los próximos 90 días.

### Señales de Fuga

- Reducción de saldo >30% en 3 meses
- Disminución de transacciones
- Cancelación de productos
- Aumento de transferencias a otros bancos
- Reducción de domiciliaciones

In [ ]:
CREATE DATABASE IF NOT EXISTS CHURN_CLIENTES_DB;
CREATE SCHEMA IF NOT EXISTS CHURN_CLIENTES_DB.PUBLIC;
USE SCHEMA CHURN_CLIENTES_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db;

---

## Paso 2: Definir Estructura y Generar Datos

### Tablas

1. **`CLIENTES_BANCA`** — 5.000 clientes con perfil
2. **`ACTIVIDAD_MENSUAL`** — 6 meses de métricas de actividad
3. **`PRODUCTOS_CLIENTE`** — Productos contratados

In [ ]:
-- Clientes
CREATE OR REPLACE TABLE CLIENTES_BANCA (
    cliente_id VARCHAR(10) PRIMARY KEY,
    segmento VARCHAR(20),
    antiguedad_anos FLOAT,
    num_productos INTEGER,
    saldo_total DECIMAL(12,2),
    ingresos_mensuales DECIMAL(10,2),
    tiene_hipoteca BOOLEAN,
    tiene_tarjeta_credito BOOLEAN,
    canal_preferido VARCHAR(20),
    es_churner BOOLEAN
);

INSERT INTO CLIENTES_BANCA
SELECT
    'CLI' || LPAD(SEQ4()::STRING, 5, '0'),
    ARRAY_CONSTRUCT('Premium','Particular','Joven','Empresa')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(1, 30, RANDOM()) + UNIFORM(0,11,RANDOM())/12.0, 1),
    UNIFORM(1, 7, RANDOM()),
    ROUND(UNIFORM(500, 200000, RANDOM()), 2),
    ROUND(UNIFORM(1000, 10000, RANDOM()), 2),
    UNIFORM(0,1,RANDOM()) < 0.35,
    UNIFORM(0,1,RANDOM()) < 0.7,
    ARRAY_CONSTRUCT('App','Web','Oficina','Telefónico')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 15 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

-- Actividad mensual (6 meses)
CREATE OR REPLACE TABLE ACTIVIDAD_MENSUAL (
    cliente_id VARCHAR(10),
    mes DATE,
    num_transacciones INTEGER,
    volumen_transacciones DECIMAL(12,2),
    num_logins INTEGER,
    transferencias_salientes INTEGER,
    domiciliaciones_activas INTEGER,
    PRIMARY KEY (cliente_id, mes)
);

INSERT INTO ACTIVIDAD_MENSUAL
SELECT
    c.cliente_id,
    DATEADD('month', -m.n, DATE_TRUNC('month', CURRENT_DATE())),
    CASE WHEN c.es_churner AND m.n <= 2 THEN UNIFORM(1, 5, RANDOM())
         ELSE UNIFORM(10, 50, RANDOM()) END,
    CASE WHEN c.es_churner AND m.n <= 2 THEN ROUND(UNIFORM(50, 500, RANDOM()), 2)
         ELSE ROUND(UNIFORM(500, 5000, RANDOM()), 2) END,
    CASE WHEN c.es_churner AND m.n <= 2 THEN UNIFORM(0, 3, RANDOM())
         ELSE UNIFORM(5, 30, RANDOM()) END,
    CASE WHEN c.es_churner AND m.n <= 2 THEN UNIFORM(3, 10, RANDOM())
         ELSE UNIFORM(0, 3, RANDOM()) END,
    CASE WHEN c.es_churner AND m.n <= 2 THEN UNIFORM(0, 2, RANDOM())
         ELSE UNIFORM(3, 10, RANDOM()) END
FROM CLIENTES_BANCA c
CROSS JOIN (SELECT SEQ4() AS n FROM TABLE(GENERATOR(ROWCOUNT => 6))) m;

SELECT es_churner, COUNT(*) FROM CLIENTES_BANCA GROUP BY 1;

---

## Paso 3: Ingeniería de Variables de Churn

### Features de Comportamiento

- **Tendencia de actividad**: Comparar últimos 2 meses vs 4 anteriores
- **Caída de saldo**: % cambio en saldo
- **Transferencias salientes**: Señal de movimiento de fondos
- **Engagement digital**: Frecuencia de uso de canales digitales

In [ ]:
-- Features de churn
CREATE OR REPLACE TABLE FEATURES_CHURN AS
WITH reciente AS (
    SELECT cliente_id,
        AVG(num_transacciones) AS txn_recientes,
        AVG(volumen_transacciones) AS vol_reciente,
        AVG(num_logins) AS logins_recientes,
        AVG(transferencias_salientes) AS transf_sal_recientes,
        AVG(domiciliaciones_activas) AS domic_recientes
    FROM ACTIVIDAD_MENSUAL WHERE mes >= DATEADD('month', -2, DATE_TRUNC('month', CURRENT_DATE())) GROUP BY 1
),
anterior AS (
    SELECT cliente_id,
        AVG(num_transacciones) AS txn_anteriores,
        AVG(volumen_transacciones) AS vol_anterior,
        AVG(num_logins) AS logins_anteriores,
        AVG(transferencias_salientes) AS transf_sal_anteriores
    FROM ACTIVIDAD_MENSUAL WHERE mes < DATEADD('month', -2, DATE_TRUNC('month', CURRENT_DATE())) GROUP BY 1
)
SELECT
    c.cliente_id,
    c.antiguedad_anos::FLOAT,
    c.num_productos::FLOAT,
    c.saldo_total::FLOAT,
    c.ingresos_mensuales::FLOAT,
    c.tiene_hipoteca::FLOAT,
    c.tiene_tarjeta_credito::FLOAT,
    COALESCE(r.txn_recientes, 0)::FLOAT AS txn_recientes,
    COALESCE(a.txn_anteriores, 0)::FLOAT AS txn_anteriores,
    CASE WHEN a.txn_anteriores > 0 THEN (r.txn_recientes - a.txn_anteriores) / a.txn_anteriores ELSE 0 END::FLOAT AS tendencia_txn,
    CASE WHEN a.vol_anterior > 0 THEN (r.vol_reciente - a.vol_anterior) / a.vol_anterior ELSE 0 END::FLOAT AS tendencia_volumen,
    COALESCE(r.logins_recientes, 0)::FLOAT AS logins_recientes,
    COALESCE(r.transf_sal_recientes, 0)::FLOAT AS transf_salientes,
    COALESCE(r.domic_recientes, 0)::FLOAT AS domiciliaciones,
    c.es_churner
FROM CLIENTES_BANCA c
LEFT JOIN reciente r ON c.cliente_id = r.cliente_id
LEFT JOIN anterior a ON c.cliente_id = a.cliente_id;

SELECT * FROM FEATURES_CHURN LIMIT 10;

---

## Paso 4: Entrenar Modelo Predictivo

### Modelo de Churn

El clasificador identificará patrones que preceden al abandono.

In [ ]:
CREATE OR REPLACE TABLE TRAIN_CHURN AS SELECT * FROM FEATURES_CHURN SAMPLE (80);
CREATE OR REPLACE TABLE TEST_CHURN AS SELECT * FROM FEATURES_CHURN WHERE cliente_id NOT IN (SELECT cliente_id FROM TRAIN_CHURN);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION predictor_churn(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_CHURN'),
    TARGET_COLNAME => 'ES_CHURNER'
);

In [ ]:
CALL predictor_churn!SHOW_EVALUATION_METRICS();
CALL predictor_churn!SHOW_FEATURE_IMPORTANCE();

---

## Paso 5: Puntuar y Recomendar Retención

### Estratificación

- **Alto Riesgo** (≥70%): Acción inmediata de retención
- **Riesgo Medio** (40-70%): Seguimiento proactivo
- **Bajo Riesgo** (<40%): Monitorización estándar

In [ ]:
-- Puntuar clientes
CREATE OR REPLACE TABLE PREDICCIONES_CHURN AS
SELECT
    cliente_id, saldo_total, num_productos, tendencia_txn, tendencia_volumen,
    transf_salientes, es_churner AS churner_real,
    predictor_churn!PREDICT(
        OBJECT_CONSTRUCT(
            'ANTIGUEDAD_ANOS', antiguedad_anos, 'NUM_PRODUCTOS', num_productos,
            'SALDO_TOTAL', saldo_total, 'INGRESOS_MENSUALES', ingresos_mensuales,
            'TIENE_HIPOTECA', tiene_hipoteca, 'TIENE_TARJETA_CREDITO', tiene_tarjeta_credito,
            'TXN_RECIENTES', txn_recientes, 'TXN_ANTERIORES', txn_anteriores,
            'TENDENCIA_TXN', tendencia_txn, 'TENDENCIA_VOLUMEN', tendencia_volumen,
            'LOGINS_RECIENTES', logins_recientes, 'TRANSF_SALIENTES', transf_salientes,
            'DOMICILIACIONES', domiciliaciones
        )
    ) AS pred,
    ROUND(pred['probability']['true']::FLOAT * 100, 1) AS prob_churn,
    CASE
        WHEN pred['probability']['true']::FLOAT >= 0.7 THEN 'Alto Riesgo'
        WHEN pred['probability']['true']::FLOAT >= 0.4 THEN 'Riesgo Medio'
        ELSE 'Bajo Riesgo'
    END AS nivel_riesgo
FROM TEST_CHURN;

SELECT nivel_riesgo, COUNT(*) AS n, ROUND(AVG(saldo_total),0) AS saldo_medio
FROM PREDICCIONES_CHURN GROUP BY 1 ORDER BY 2;

---

## Paso 6: Recomendaciones IA de Retención

Usamos `CORTEX.COMPLETE()` para generar acciones de retención personalizadas.

In [ ]:
CREATE OR REPLACE TABLE RECOMENDACIONES_RETENCION AS
SELECT
    cliente_id, nivel_riesgo, prob_churn, saldo_total, num_productos,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Eres un gestor de retención bancaria. Genera 3 acciones concretas para retener a este cliente:\n' ||
        'Riesgo de fuga: ' || prob_churn || '%\n' ||
        'Saldo: ' || saldo_total || '€\n' ||
        'Productos: ' || num_productos || '\n' ||
        'Tendencia transacciones: ' || tendencia_txn || '\n' ||
        'Responde solo con las 3 acciones en español, una por línea.'
    ) AS acciones_retencion
FROM PREDICCIONES_CHURN
WHERE nivel_riesgo = 'Alto Riesgo'
SAMPLE (30 ROWS);

SELECT * FROM RECOMENDACIONES_RETENCION LIMIT 5;

---

## Paso 7: Dashboard de Retención

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Predicción de Fuga de Clientes')
st.markdown('### Panel de Retención')

total = session.sql('SELECT COUNT(*) FROM PREDICCIONES_CHURN').collect()[0][0]
alto = session.sql("SELECT COUNT(*) FROM PREDICCIONES_CHURN WHERE nivel_riesgo='Alto Riesgo'").collect()[0][0]
saldo_riesgo = session.sql("SELECT ROUND(SUM(saldo_total),0) FROM PREDICCIONES_CHURN WHERE nivel_riesgo='Alto Riesgo'").collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Clientes', f'{total:,}')
c2.metric('Alto Riesgo', f'{alto:,}')
c3.metric('Saldo en Riesgo', f'{saldo_riesgo:,.0f}€')

st.markdown('---')
df = session.sql('SELECT nivel_riesgo, COUNT(*) AS n FROM PREDICCIONES_CHURN GROUP BY 1').to_pandas()
st.bar_chart(df.set_index('NIVEL_RIESGO'))

st.markdown('---')
st.subheader('Clientes con Recomendaciones de Retención')
df_ret = session.sql('SELECT cliente_id, prob_churn, saldo_total, acciones_retencion FROM RECOMENDACIONES_RETENCION ORDER BY prob_churn DESC LIMIT 15').to_pandas()
st.dataframe(df_ret)

---

## Paso 8: Limpieza (Opcional)

⚠️ Eliminará todos los datos y modelos.

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS CHURN_CLIENTES_DB;